# PathSnap MLP — Teammate Prepared Dataset

This notebook trains and evaluates the **new four-feature MLP** using the teammate-prepared candidate tables and their frozen five-fold trip split. It is kept separate from the original PathSnap MLP notebook and model.

The supplied candidate tables omit observations where the reference candidate is absent. Consequently candidate-level metrics and conditional Top-1 are conditional on a reference-hit candidate group; end-to-end Top-1 counts omitted aligned observations as misses. See `RESULTS.md` for the dataset audit and caveats.

## 1. Locate files and load dependencies

Keep this notebook in `teammate_mlp_results` inside the delivered PathSnap package. The prepared tables and training script are in the parent package folder.

In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd

HERE = Path.cwd().resolve()
if (HERE / "teammate_dataset").is_dir():
    PACKAGE_DIR = HERE
elif (HERE.parent / "teammate_dataset").is_dir():
    PACKAGE_DIR = HERE.parent
else:
    candidates = [p for p in HERE.parents if (p / "teammate_dataset").is_dir()]
    if not candidates:
        raise FileNotFoundError("Could not find teammate_dataset. Keep the notebook inside the delivered package.")
    PACKAGE_DIR = candidates[0]
DATA_DIR = PACKAGE_DIR / "teammate_dataset"
RESULTS_DIR = PACKAGE_DIR / "teammate_mlp_results"
FEATURES = ["distance_to_segment_m", "projection_fraction",
            "observation_speed_mps", "heading_difference_rad"]
print("Package:", PACKAGE_DIR)
print("Dataset:", DATA_DIR)
print("Results:", RESULTS_DIR)

## 2. Inspect the provided dataset

IDs and reference labels are used only for grouping and evaluation. The model predictors are exactly the four numeric features shown above.

In [ ]:
train = pd.read_csv(DATA_DIR / "train_candidates.csv")
test = pd.read_csv(DATA_DIR / "student_test_candidates.csv")
train_coverage = pd.read_csv(DATA_DIR / "train_coverage.csv")
test_coverage = pd.read_csv(DATA_DIR / "student_test_coverage.csv")
fold_cfg = json.loads((DATA_DIR / "fold_assignments.json").read_text(encoding="utf-8"))

def audit_table(name, candidates, coverage):
    groups = candidates.groupby(["trip_id", "observation_id"], sort=False)
    n_groups = groups.ngroups
    actual = candidates.groupby(["trip_id", "observation_id"]).size().rename("actual").reset_index()
    counts = coverage.merge(actual, on=["trip_id", "observation_id"], how="left")
    counts["actual"] = counts["actual"].fillna(0).astype(int)
    print(f"{name}: {len(candidates):,} candidate rows; {candidates.trip_id.nunique()} trips; {n_groups:,} represented observations")
    print("  missing/nonfinite features:", int(candidates[FEATURES].isna().sum().sum()), int((~np.isfinite(candidates[FEATURES].to_numpy())).sum()))
    print("  labels:", candidates.label.value_counts().sort_index().to_dict())
    print("  duplicate candidate rows:", int(candidates.duplicated(["trip_id", "observation_id", "candidate_id"]).sum()))
    print("  coverage observations with candidate rows missing:", int((counts.actual == 0).sum()))
    print("  coverage num_candidates mismatches:", int((counts.actual != counts.num_candidates).sum()))
    print("  aligned observations / reference candidates present:", int(coverage.is_aligned.sum()), int(coverage.ref_in_candidates.sum()))

audit_table("Train", train, train_coverage)
audit_table("Student test", test, test_coverage)

## 3. Verify frozen trip folds

The JSON stores trip IDs as zero-padded strings while the CSV reader may infer numeric IDs, so normalize CSV IDs to eight-character strings before comparing.

In [ ]:
folds = fold_cfg["folds"]
all_trips = set(train.trip_id.map(lambda v: f"{int(v):08d}"))
validation_union = set()
for key, split in folds.items():
    train_trips, val_trips = set(split["train_trips"]), set(split["val_trips"])
    assert not (train_trips & val_trips), f"Overlap in fold {key}"
    assert train_trips == all_trips - val_trips, f"Unexpected train/validation membership in fold {key}"
    validation_union |= val_trips
    print(f"Fold {key}: {len(train_trips)} train trips, {len(val_trips)} validation trips")
assert validation_union == all_trips
print("All 80 training trips are assigned to validation exactly once.")

## 4. Train, cross-validate, and evaluate

This calls the package training script, which scales features within each training fold, fits the `(32, 16)` hidden-layer MLP on the provided five folds, refits on all training trips, scores the held-out student-test trips, and saves the model, predictions, metrics, and loss histories. Rerunning this cell retrains and replaces those result files.

In [ ]:
import importlib.util
script = PACKAGE_DIR / "train_teammate_mlp.py"
spec = importlib.util.spec_from_file_location("train_teammate_mlp", script)
trainer = importlib.util.module_from_spec(spec)
spec.loader.exec_module(trainer)
report = trainer.main()

## 5. Review saved scores

The following cell displays the held-out scores and the five-fold mean with fold-to-fold standard deviation.

In [ ]:
metrics = json.loads((RESULTS_DIR / "metrics.json").read_text(encoding="utf-8"))
cv = metrics["cross_validation_mean_and_sd"]
test_metrics = metrics["student_test_metrics"]
summary = pd.DataFrame([
    {"Evaluation": "5-fold mean (SD)", "Candidate accuracy": f"{cv['candidate_accuracy']['mean']:.3f} ({cv['candidate_accuracy']['std']:.3f})",
     "Macro F1": f"{cv['macro_f1']['mean']:.3f} ({cv['macro_f1']['std']:.3f})",
     "ROC AUC": f"{cv['roc_auc']['mean']:.3f} ({cv['roc_auc']['std']:.3f})",
     "End-to-end Top-1": f"{cv['top1_accuracy_end_to_end_over_aligned_observations']['mean']:.3f} ({cv['top1_accuracy_end_to_end_over_aligned_observations']['std']:.3f})"},
    {"Evaluation": "Student test", "Candidate accuracy": f"{test_metrics['candidate_accuracy']:.3f}",
     "Macro F1": f"{test_metrics['macro_f1']:.3f}", "ROC AUC": f"{test_metrics['roc_auc']:.3f}",
     "End-to-end Top-1": f"{test_metrics['top1_accuracy_end_to_end_over_aligned_observations']:.3f}"}
])
display(summary)
print("Saved model:", RESULTS_DIR / "artifacts" / "pathsnap_mlp.joblib")
print("Full written report:", RESULTS_DIR / "RESULTS.md")

## Model summary

- Four inputs: distance to segment, projection along the segment, observation speed, and heading difference.
- Architecture: 4 → 32 ReLU → 16 ReLU → 1 sigmoid.
- StandardScaler is fit using only the training partition in each fold.
- The separate held-out student-test set is used only for final evaluation.
- Route continuity is unavailable from this archive because road-segment connectivity is not included.

See `RESULTS.md` and `metrics.json` for the full audit, scores, and limitations.